# Session 1: Introduction to Polars

Welcome to the Advanced Tech Track! In this course, we'll learn **Polars**, a modern DataFrame library that offers significant performance advantages over Pandas.

## Learning Objectives

By the end of this session, you will be able to:
1. Understand what Polars is and why it's useful
2. Create DataFrames and Series in Polars
3. Read and write various file formats
4. Perform basic data inspection
5. Select and transform columns using the expression API

## Prerequisites

This course assumes you're familiar with:
- Python fundamentals
- Pandas basics (DataFrames, Series, filtering, groupby)

## 1. What is Polars?

**Polars** is a DataFrame library written in Rust with Python bindings. It's designed for:

- **Speed**: Often 10-100x faster than Pandas for large datasets
- **Memory efficiency**: Better memory management and lazy evaluation
- **Modern API**: Consistent, expressive syntax based on expressions
- **Parallel execution**: Automatic parallelization of operations

### Why learn Polars?

| Aspect | Pandas | Polars |
|--------|--------|--------|
| Written in | C/Cython | Rust |
| Memory model | Eager only | Eager + Lazy |
| Parallelization | Manual | Automatic |
| Index | Row index | No index |
| Missing values | NaN + None | null |
| String handling | object dtype | Native strings |

In [1]:
# Import Polars
import polars as pl

# Check version
pl.__version__

Polars version: 1.37.1


## 2. Creating DataFrames and Series

Let's start by creating DataFrames - the core data structure in Polars.

### 2.1 Creating a DataFrame from a dictionary

In [2]:
# Create a DataFrame from a dictionary
df = pl.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "age": [25, 30, 35, 28],
    "city": ["New York", "Paris", "London", "Tokyo"]
})

df

name,age,city
str,i64,str
"""Alice""",25,"""New York"""
"""Bob""",30,"""Paris"""
"""Charlie""",35,"""London"""
"""Diana""",28,"""Tokyo"""


### Pandas Comparison

```python
# Pandas
import pandas as pd
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "age": [25, 30, 35, 28],
    "city": ["New York", "Paris", "London", "Tokyo"]
})
```

The syntax is nearly identical! The main difference is `pl.DataFrame` vs `pd.DataFrame`.

### 2.2 Creating a Series

In [3]:
# Create a Series
s = pl.Series("temperatures", [22.5, 25.0, 18.3, 30.1, 27.8])
s

shape: (5,)
Series: 'temperatures' [f64]
[
	22.5
	25.0
	18.3
	30.1
	27.8
]

Data type: Float64


In [ ]:
# Check the data type
s.dtype

In [4]:
# Series with different data types
dates = pl.Series("dates", ["2024-01-01", "2024-01-02", "2024-01-03"]).str.to_date()
dates

shape: (3,)
Series: 'dates' [date]
[
	2024-01-01
	2024-01-02
	2024-01-03
]

Data type: Date


In [ ]:
dates.dtype

### Pandas Comparison: Series

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Create Series | `pd.Series([1, 2, 3])` | `pl.Series([1, 2, 3])` |
| Named Series | `pd.Series([1, 2, 3], name="values")` | `pl.Series("values", [1, 2, 3])` |
| Access dtype | `s.dtype` | `s.dtype` |
| To datetime | `pd.to_datetime(s)` | `s.str.to_date()` |

**Note**: In Polars, the series name comes *first* in the constructor: `pl.Series("name", [values])`, while in Pandas it's a keyword argument: `pd.Series([values], name="name")`.

## 3. Reading and Writing Files

Polars supports many file formats. Let's explore the most common ones.

### 3.1 Reading CSV Files

In [5]:
# Read a CSV file
employees = pl.read_csv("data/employees.csv")
employees.head()

employee_id,first_name,last_name,email,department,position,salary,hire_date,is_active
i64,str,str,str,str,str,i64,str,bool
1,"""Noah""","""Smith""","""employee1@company.com""","""Operations""","""Business Analyst""",77098,"""2020-07-03""",true
2,"""Michael""","""Moore""","""employee2@company.com""","""Engineering""","""Software Engineer""",122397,"""2022-09-25""",true
3,"""Sophia""","""Davis""","""employee3@company.com""","""Engineering""","""Tech Lead""",123907,"""2018-04-19""",true
4,"""Luna""","""Moore""","""employee4@company.com""","""Operations""","""Process Engineer""",73893,"""2023-01-14""",true
5,"""Gianna""","""Garcia""","""employee5@company.com""","""Engineering""","""QA Engineer""",89597,"""2021-02-12""",true


### Pandas Comparison

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Read CSV | `pd.read_csv()` | `pl.read_csv()` |
| Read JSON | `pd.read_json()` | `pl.read_json()` |
| Read Parquet | `pd.read_parquet()` | `pl.read_parquet()` |
| Read Excel | `pd.read_excel()` | `pl.read_excel()` |

### 3.2 Writing Files

In [6]:
# Write to CSV
employees.head(10).write_csv("data/employees_sample.csv")

# Write to Parquet (efficient columnar format)
employees.write_parquet("data/employees.parquet")

Files written successfully!


In [7]:
# Read back the Parquet file
employees_parquet = pl.read_parquet("data/employees.parquet")
employees_parquet.head()

employee_id,first_name,last_name,email,department,position,salary,hire_date,is_active
i64,str,str,str,str,str,i64,str,bool
1,"""Noah""","""Smith""","""employee1@company.com""","""Operations""","""Business Analyst""",77098,"""2020-07-03""",true
2,"""Michael""","""Moore""","""employee2@company.com""","""Engineering""","""Software Engineer""",122397,"""2022-09-25""",true
3,"""Sophia""","""Davis""","""employee3@company.com""","""Engineering""","""Tech Lead""",123907,"""2018-04-19""",true
4,"""Luna""","""Moore""","""employee4@company.com""","""Operations""","""Process Engineer""",73893,"""2023-01-14""",true
5,"""Gianna""","""Garcia""","""employee5@company.com""","""Engineering""","""QA Engineer""",89597,"""2021-02-12""",true


### 3.3 Understanding Parquet Format

**Parquet** is a columnar storage format designed for efficient data storage and retrieval. Unlike CSV (which stores data row by row), Parquet stores data column by column.

#### CSV vs Parquet: How They Store Data

```
CSV (Row-oriented):          Parquet (Column-oriented):
┌────┬─────┬──────┐          ┌────────────────────┐
│ id │ name│ sal  │          │ id: 1, 2, 3        │
├────┼─────┼──────┤          │ name: A, B, C      │
│ 1  │ A   │ 50k  │          │ sal: 50k, 60k, 70k │
│ 2  │ B   │ 60k  │          └────────────────────┘
│ 3  │ C   │ 70k  │
└────┴─────┴──────┘
```

#### Why Parquet is Better for Analytics

| Feature | CSV | Parquet |
|---------|-----|---------|
| **Storage** | Plain text | Binary, compressed |
| **File Size** | Large | 2-10x smaller |
| **Data Types** | Lost (everything is text) | Preserved |
| **Read Speed** | Must parse text | Direct binary read |
| **Column Selection** | Must read entire file | Reads only needed columns |
| **Schema** | None (inferred) | Embedded in file |

#### When to Use Each Format

- **CSV**: Human-readable, data exchange, small datasets, compatibility
- **Parquet**: Analytics, large datasets, repeated reads, data pipelines

### 3.4 Performance Comparison: Pandas vs Polars, CSV vs Parquet

Let's compare the read performance of different combinations.

In [8]:
import pandas as pd
import time
import os

# First, let's check file sizes
csv_path = "data/employees.csv"
parquet_path = "data/employees.parquet"

csv_size = os.path.getsize(csv_path)
parquet_size = os.path.getsize(parquet_path)

pl.DataFrame({
    "format": ["CSV", "Parquet"],
    "size_bytes": [csv_size, parquet_size],
    "ratio": [f"1x", f"{csv_size/parquet_size:.1f}x smaller"]
})

=== File Size Comparison ===
CSV file:     8,762 bytes
Parquet file: 5,726 bytes
Parquet is 1.5x smaller


In [9]:
# Benchmark function
def benchmark_read(read_func, path, n_runs=10):
    """Run read operation multiple times and return average time."""
    times = []
    for _ in range(n_runs):
        start = time.time()
        df = read_func(path)
        times.append(time.time() - start)
    return sum(times) / len(times) * 1000  # Return milliseconds

# Run benchmarks (10 runs each)
pandas_csv_time = benchmark_read(pd.read_csv, csv_path)
pandas_parquet_time = benchmark_read(pd.read_parquet, parquet_path)
polars_csv_time = benchmark_read(pl.read_csv, csv_path)
polars_parquet_time = benchmark_read(pl.read_parquet, parquet_path)

pl.DataFrame({
    "library + format": ["Pandas + CSV", "Pandas + Parquet", "Polars + CSV", "Polars + Parquet"],
    "avg_ms": [pandas_csv_time, pandas_parquet_time, polars_csv_time, polars_parquet_time],
    "speedup_vs_pandas_csv": [
        "1.0x (baseline)",
        f"{pandas_csv_time/pandas_parquet_time:.1f}x",
        f"{pandas_csv_time/polars_csv_time:.1f}x",
        f"{pandas_csv_time/polars_parquet_time:.1f}x",
    ]
})

=== Read Performance Comparison (10 runs each) ===

Pandas  + CSV:     0.95 ms
Pandas  + Parquet: 29.03 ms
Polars  + CSV:     2.19 ms
Polars  + Parquet: 0.41 ms

=== Speedup Summary ===
Polars CSV vs Pandas CSV:         0.4x faster
Polars Parquet vs Pandas Parquet: 70.9x faster
Polars Parquet vs Pandas CSV:     2.3x faster


### 3.5 Scaling Up: Performance with Larger Datasets

The 100-row dataset above is too small to show meaningful differences. Let's generate a **500,000-row** dataset and see how the performance gap widens.

In [ ]:
import random
import string

random.seed(42)
n_rows = 500_000

departments = ["Engineering", "Marketing", "Sales", "HR", "Finance", "Operations", "Legal", "Support"]
positions = ["Junior", "Mid", "Senior", "Lead", "Manager", "Director", "VP"]

large_df = pl.DataFrame({
    "employee_id": range(1, n_rows + 1),
    "first_name": [f"Employee_{i}" for i in range(n_rows)],
    "last_name": ["".join(random.choices(string.ascii_uppercase, k=6)) for _ in range(n_rows)],
    "department": [random.choice(departments) for _ in range(n_rows)],
    "position": [random.choice(positions) for _ in range(n_rows)],
    "salary": [random.randint(35_000, 180_000) for _ in range(n_rows)],
    "years_experience": [random.randint(0, 30) for _ in range(n_rows)],
    "performance_score": [round(random.uniform(1.0, 5.0), 2) for _ in range(n_rows)],
})

# Write both formats
large_df.write_csv("data/employees_large.csv")
large_df.write_parquet("data/employees_large.parquet")

large_df.head(3)

In [ ]:
# Compare file sizes
large_csv_size = os.path.getsize("data/employees_large.csv")
large_parquet_size = os.path.getsize("data/employees_large.parquet")

pl.DataFrame({
    "format": ["CSV", "Parquet"],
    "size_MB": [round(large_csv_size / 1e6, 2), round(large_parquet_size / 1e6, 2)],
    "compression": ["1x (baseline)", f"{large_csv_size / large_parquet_size:.1f}x smaller"],
})

In [ ]:
# Benchmark on the large dataset
large_csv_path = "data/employees_large.csv"
large_parquet_path = "data/employees_large.parquet"

lg_pandas_csv = benchmark_read(pd.read_csv, large_csv_path)
lg_pandas_parquet = benchmark_read(pd.read_parquet, large_parquet_path)
lg_polars_csv = benchmark_read(pl.read_csv, large_csv_path)
lg_polars_parquet = benchmark_read(pl.read_parquet, large_parquet_path)

pl.DataFrame({
    "library + format": ["Pandas + CSV", "Pandas + Parquet", "Polars + CSV", "Polars + Parquet"],
    "avg_ms": [
        round(lg_pandas_csv, 1),
        round(lg_pandas_parquet, 1),
        round(lg_polars_csv, 1),
        round(lg_polars_parquet, 1),
    ],
    "speedup_vs_pandas_csv": [
        "1.0x (baseline)",
        f"{lg_pandas_csv / lg_pandas_parquet:.1f}x",
        f"{lg_pandas_csv / lg_polars_csv:.1f}x",
        f"{lg_pandas_csv / lg_polars_parquet:.1f}x",
    ],
})

### Key Takeaways: File Formats and Performance

1. **Parquet files are smaller** due to efficient binary compression (often 3-10x)
2. **Parquet reads are faster** because:
   - No text parsing required
   - Data types don't need inference
   - Can read only needed columns (projection pushdown)
3. **Polars is faster than Pandas** for both CSV and Parquet formats
4. **Best combination**: Polars + Parquet for maximum performance
5. **The gap grows with data size**: At 500K rows the Polars + Parquet advantage becomes dramatic

**Recommendation**: When working with data you'll read multiple times, convert CSV to Parquet:

```python
# One-time conversion
pl.read_csv("data.csv").write_parquet("data.parquet")

# Then always read from Parquet
df = pl.read_parquet("data.parquet")
```

## 4. Basic Data Inspection

Let's explore how to inspect our data in Polars.

In [10]:
# Shape: (rows, columns)
employees.shape

Shape: (100, 9)
Rows: 100
Columns: 9


In [ ]:
employees.height, employees.width

In [11]:
# Column names
employees.columns

Columns: ['employee_id', 'first_name', 'last_name', 'email', 'department', 'position', 'salary', 'hire_date', 'is_active']


In [12]:
# Data types
employees.dtypes

Data types:
[Int64, String, String, String, String, String, Int64, String, Boolean]


In [13]:
# Schema (column name -> data type mapping)
employees.schema

Schema([('employee_id', Int64),
        ('first_name', String),
        ('last_name', String),
        ('email', String),
        ('department', String),
        ('position', String),
        ('salary', Int64),
        ('hire_date', String),
        ('is_active', Boolean)])

In [14]:
# First n rows
employees.head(5)

employee_id,first_name,last_name,email,department,position,salary,hire_date,is_active
i64,str,str,str,str,str,i64,str,bool
1,"""Noah""","""Smith""","""employee1@company.com""","""Operations""","""Business Analyst""",77098,"""2020-07-03""",true
2,"""Michael""","""Moore""","""employee2@company.com""","""Engineering""","""Software Engineer""",122397,"""2022-09-25""",true
3,"""Sophia""","""Davis""","""employee3@company.com""","""Engineering""","""Tech Lead""",123907,"""2018-04-19""",true
4,"""Luna""","""Moore""","""employee4@company.com""","""Operations""","""Process Engineer""",73893,"""2023-01-14""",true
5,"""Gianna""","""Garcia""","""employee5@company.com""","""Engineering""","""QA Engineer""",89597,"""2021-02-12""",true


In [15]:
# Last n rows
employees.tail(5)

employee_id,first_name,last_name,email,department,position,salary,hire_date,is_active
i64,str,str,str,str,str,i64,str,bool
96,"""Ava""","""Anderson""","""employee96@company.com""","""HR""","""Recruiter""",51590,"""2020-12-01""",true
97,"""Amelia""","""Hernandez""","""employee97@company.com""","""Marketing""","""SEO Specialist""",144775,"""2022-04-02""",true
98,"""Isabella""","""Williams""","""employee98@company.com""","""HR""","""Training Specialist""",47540,"""2024-01-19""",true
99,"""Mason""","""Williams""","""employee99@company.com""","""Sales""","""Sales Rep""",143858,"""2018-05-08""",false
100,"""Abigail""","""Smith""","""employee100@company.com""","""Marketing""","""Digital Marketer""",64973,"""2020-09-04""",true


In [16]:
# Statistical summary
employees.describe()

statistic,employee_id,first_name,last_name,email,department,position,salary,hire_date,is_active
str,f64,str,str,str,str,str,f64,str,f64
"""count""",100.0,"""100""","""100""","""100""","""100""","""100""",100.0,"""100""",100.0
"""null_count""",0.0,"""0""","""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",50.5,null,null,null,null,null,101081.12,null,0.84
"""std""",29.011492,null,null,null,null,null,30622.548451,null,null
"""min""",1.0,"""Abigail""","""Anderson""","""employee100@company.com""","""Engineering""","""Account Manager""",46934.0,"""2018-01-15""",0.0
"""25%""",26.0,null,null,null,null,null,75311.0,null,null
"""50%""",51.0,null,null,null,null,null,106261.0,null,null
"""75%""",75.0,null,null,null,null,null,126351.0,null,null
"""max""",100.0,"""William""","""Wilson""","""employee9@company.com""","""Sales""","""Treasury Analyst""",147056.0,"""2024-12-08""",1.0


### Pandas Comparison: Inspection Methods

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Shape | `df.shape` | `df.shape` |
| Columns | `df.columns` | `df.columns` |
| Data types | `df.dtypes` | `df.dtypes` |
| First rows | `df.head()` | `df.head()` |
| Last rows | `df.tail()` | `df.tail()` |
| Summary | `df.describe()` | `df.describe()` |
| Info | `df.info()` | `df.schema` |

## 5. Column Selection with `select()` and `pl.col()`

This is where Polars starts to differ from Pandas. Polars uses an **expression API** for selecting and transforming columns.

### 5.1 Basic Column Selection

In [17]:
# Select a single column by name (returns DataFrame)
employees.select("first_name")

first_name
str
"""Noah"""
"""Michael"""
"""Sophia"""
"""Luna"""
"""Gianna"""
…
"""Ava"""
"""Amelia"""
"""Isabella"""


In [18]:
# Select multiple columns
employees.select("first_name", "last_name", "department")

first_name,last_name,department
str,str,str
"""Noah""","""Smith""","""Operations"""
"""Michael""","""Moore""","""Engineering"""
"""Sophia""","""Davis""","""Engineering"""
"""Luna""","""Moore""","""Operations"""
"""Gianna""","""Garcia""","""Engineering"""
…,…,…
"""Ava""","""Anderson""","""HR"""
"""Amelia""","""Hernandez""","""Marketing"""
"""Isabella""","""Williams""","""HR"""


In [19]:
# Using pl.col() - the expression way
employees.select(pl.col("first_name"), pl.col("salary"))

first_name,salary
str,i64
"""Noah""",77098
"""Michael""",122397
"""Sophia""",123907
"""Luna""",73893
"""Gianna""",89597
…,…
"""Ava""",51590
"""Amelia""",144775
"""Isabella""",47540


### 5.2 Selecting with Patterns

In [20]:
# Select all columns
employees.select(pl.all())

employee_id,first_name,last_name,email,department,position,salary,hire_date,is_active
i64,str,str,str,str,str,i64,str,bool
1,"""Noah""","""Smith""","""employee1@company.com""","""Operations""","""Business Analyst""",77098,"""2020-07-03""",true
2,"""Michael""","""Moore""","""employee2@company.com""","""Engineering""","""Software Engineer""",122397,"""2022-09-25""",true
3,"""Sophia""","""Davis""","""employee3@company.com""","""Engineering""","""Tech Lead""",123907,"""2018-04-19""",true
4,"""Luna""","""Moore""","""employee4@company.com""","""Operations""","""Process Engineer""",73893,"""2023-01-14""",true
5,"""Gianna""","""Garcia""","""employee5@company.com""","""Engineering""","""QA Engineer""",89597,"""2021-02-12""",true
…,…,…,…,…,…,…,…,…
96,"""Ava""","""Anderson""","""employee96@company.com""","""HR""","""Recruiter""",51590,"""2020-12-01""",true
97,"""Amelia""","""Hernandez""","""employee97@company.com""","""Marketing""","""SEO Specialist""",144775,"""2022-04-02""",true
98,"""Isabella""","""Williams""","""employee98@company.com""","""HR""","""Training Specialist""",47540,"""2024-01-19""",true


In [21]:
# Select columns that start with a pattern
employees.select(pl.col("^.*_name$"))  # Columns ending with '_name'

first_name,last_name
str,str
"""Noah""","""Smith"""
"""Michael""","""Moore"""
"""Sophia""","""Davis"""
"""Luna""","""Moore"""
"""Gianna""","""Garcia"""
…,…
"""Ava""","""Anderson"""
"""Amelia""","""Hernandez"""
"""Isabella""","""Williams"""


In [22]:
# Select columns by data type
employees.select(pl.col(pl.Int64))  # Only integer columns

employee_id,salary
i64,i64
1,77098
2,122397
3,123907
4,73893
5,89597
…,…
96,51590
97,144775
98,47540


### Pandas Comparison: Pattern-Based Selection

| Operation | Pandas | Polars |
|-----------|--------|--------|
| All columns | `df` or `df.loc[:, :]` | `df.select(pl.all())` |
| Regex pattern | `df.filter(regex=r"^.*_name$")` | `df.select(pl.col("^.*_name$"))` |
| By dtype | `df.select_dtypes(include=['int64'])` | `df.select(pl.col(pl.Int64))` |
| String columns | `df.select_dtypes(include=['object'])` | `df.select(pl.col(pl.String))` |
| Exclude columns | `df.drop(columns=["col"])` | `df.select(pl.all().exclude("col"))` |

**Note**: Polars' `pl.col()` accepts regex patterns directly, making pattern-based selection more concise than Pandas' `filter()` method.

In [23]:
# Select all string columns
employees.select(pl.col(pl.String))

first_name,last_name,email,department,position,hire_date
str,str,str,str,str,str
"""Noah""","""Smith""","""employee1@company.com""","""Operations""","""Business Analyst""","""2020-07-03"""
"""Michael""","""Moore""","""employee2@company.com""","""Engineering""","""Software Engineer""","""2022-09-25"""
"""Sophia""","""Davis""","""employee3@company.com""","""Engineering""","""Tech Lead""","""2018-04-19"""
"""Luna""","""Moore""","""employee4@company.com""","""Operations""","""Process Engineer""","""2023-01-14"""
"""Gianna""","""Garcia""","""employee5@company.com""","""Engineering""","""QA Engineer""","""2021-02-12"""
…,…,…,…,…,…
"""Ava""","""Anderson""","""employee96@company.com""","""HR""","""Recruiter""","""2020-12-01"""
"""Amelia""","""Hernandez""","""employee97@company.com""","""Marketing""","""SEO Specialist""","""2022-04-02"""
"""Isabella""","""Williams""","""employee98@company.com""","""HR""","""Training Specialist""","""2024-01-19"""


### Pandas Comparison: Column Selection

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Single column | `df["col"]` or `df.col` | `df.select("col")` |
| Multiple columns | `df[["col1", "col2"]]` | `df.select("col1", "col2")` |
| All columns | `df` | `df.select(pl.all())` |
| By dtype | `df.select_dtypes(include=['int64'])` | `df.select(pl.col(pl.Int64))` |

## 6. Introduction to the Expression API

The **expression API** is the heart of Polars and what makes it fundamentally different from Pandas.

### What is an Expression?

An **expression** in Polars is a *description of a computation*, not the result itself. Think of it like a recipe:
- The recipe (expression) describes what to do
- The cooking (execution) happens later when you call `.select()`, `.with_columns()`, or `.collect()`

This separation allows Polars to:
1. **Optimize** the computation before running it
2. **Parallelize** operations automatically
3. **Reuse** expressions across different contexts

### Pandas vs Polars: Mental Model

| Aspect | Pandas | Polars |
|--------|--------|--------|
| Column access | Direct: `df["col"]` returns data | Expression: `pl.col("col")` returns a recipe |
| When computed | Immediately | When collected/selected |
| Optimization | None (eager) | Query optimization possible |
| Reusability | Limited | Expressions can be stored and reused |

### 6.1 Basic Expressions

In [24]:
# Expressions can be stored in variables and reused
salary_with_raise = (pl.col("salary") * 1.1).alias("salary_raised")
salary_with_bonus = (pl.col("salary") * 1.2).alias("salary_with_bonus")

# Use the stored expressions
employees.select(
    pl.col("first_name"),
    pl.col("salary"),
    salary_with_raise,
    salary_with_bonus
)

first_name,salary,salary_raised,salary_with_bonus
str,i64,f64,f64
"""Noah""",77098,84807.8,92517.6
"""Michael""",122397,134636.7,146876.4
"""Sophia""",123907,136297.7,148688.4
"""Luna""",73893,81282.3,88671.6
"""Gianna""",89597,98556.7,107516.4
…,…,…,…
"""Ava""",51590,56749.0,61908.0
"""Amelia""",144775,159252.5,173730.0
"""Isabella""",47540,52294.0,57048.0


In [25]:
# Expressions can chain multiple operations
employees.select(
    pl.col("first_name"),
    pl.col("salary"),
    (pl.col("salary") / 12).round(2).alias("monthly_salary")  # Chain division and rounding
)

first_name,salary,monthly_salary
str,i64,f64
"""Noah""",77098,6424.83
"""Michael""",122397,10199.75
"""Sophia""",123907,10325.58
"""Luna""",73893,6157.75
"""Gianna""",89597,7466.42
…,…,…
"""Ava""",51590,4299.17
"""Amelia""",144775,12064.58
"""Isabella""",47540,3961.67


In [26]:
# The key difference: pl.col() vs df["col"]

# In Pandas, this immediately accesses the data:
# pandas_salary = df["salary"]  # Returns actual data (Series)

# In Polars, this creates an expression (a recipe):
polars_salary_expr = pl.col("salary")  # Returns an expression object
polars_salary_expr

This is an expression object: <class 'polars.expr.expr.Expr'>
Expression: col("salary")

After select(), we get actual data: <class 'polars.dataframe.frame.DataFrame'>


In [ ]:
# The expression is only evaluated when used in a context like select()
employees.select(polars_salary_expr)

### Pandas Comparison: Column Access vs Expressions

```python
# PANDAS - Direct access, immediate execution
salary_data = df["salary"]           # Returns Series with actual values
doubled = df["salary"] * 2           # Computed immediately
df["doubled"] = df["salary"] * 2     # Adds column immediately

# POLARS - Expression-based, deferred execution  
salary_expr = pl.col("salary")              # Returns expression (recipe)
doubled_expr = pl.col("salary") * 2         # Still just an expression
df.with_columns((pl.col("salary") * 2).alias("doubled"))  # Executed here
```

**Key insight**: In Polars, you build up expressions that describe what you want, then execute them all at once. This allows Polars to optimize the entire operation.

### 6.2 Arithmetic Expressions

Polars supports all standard arithmetic operations on expressions.

In [27]:
# Arithmetic operations on expressions
employees.select(
    pl.col("first_name"),
    pl.col("salary"),
    (pl.col("salary") + 5000).alias("salary_plus_5k"),       # Addition
    (pl.col("salary") - 10000).alias("salary_minus_10k"),    # Subtraction
    (pl.col("salary") * 2).alias("salary_doubled"),          # Multiplication
    (pl.col("salary") / 1000).alias("salary_in_thousands"),  # Division
    (pl.col("salary") // 10000).alias("salary_10k_brackets"), # Floor division
    (pl.col("salary") % 10000).alias("salary_mod_10k"),      # Modulo
)

first_name,salary,salary_plus_5k,salary_minus_10k,salary_doubled,salary_in_thousands,salary_10k_brackets,salary_mod_10k
str,i64,i64,i64,i64,f64,i64,i64
"""Noah""",77098,82098,67098,154196,77.098,7,7098
"""Michael""",122397,127397,112397,244794,122.397,12,2397
"""Sophia""",123907,128907,113907,247814,123.907,12,3907
"""Luna""",73893,78893,63893,147786,73.893,7,3893
"""Gianna""",89597,94597,79597,179194,89.597,8,9597
…,…,…,…,…,…,…,…
"""Ava""",51590,56590,41590,103180,51.59,5,1590
"""Amelia""",144775,149775,134775,289550,144.775,14,4775
"""Isabella""",47540,52540,37540,95080,47.54,4,7540


### Pandas Comparison: Arithmetic Operations

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Addition | `df["col"] + 5000` | `pl.col("col") + 5000` |
| Subtraction | `df["col"] - 1000` | `pl.col("col") - 1000` |
| Multiplication | `df["col"] * 2` | `pl.col("col") * 2` |
| Division | `df["col"] / 1000` | `pl.col("col") / 1000` |
| Floor division | `df["col"] // 10` | `pl.col("col") // 10` |
| Modulo | `df["col"] % 10` | `pl.col("col") % 10` |
| Round | `df["col"].round(2)` | `pl.col("col").round(2)` |
| Absolute | `df["col"].abs()` | `pl.col("col").abs()` |

The operators are identical! The difference is that Pandas operates on data directly, while Polars builds an expression.

### 6.3 Aggregation Expressions

In [28]:
# Compute aggregations
employees.select(
    pl.col("salary").mean().alias("avg_salary"),
    pl.col("salary").min().alias("min_salary"),
    pl.col("salary").max().alias("max_salary"),
    pl.col("salary").std().alias("std_salary")
)

avg_salary,min_salary,max_salary,std_salary
f64,i64,i64,f64
101081.12,46934,147056,30622.548451


In [29]:
# Count unique values
employees.select(
    pl.col("department").n_unique().alias("unique_departments"),
    pl.col("position").n_unique().alias("unique_positions")
)

unique_departments,unique_positions
u32,u32
6,29


### Pandas Comparison: Aggregation Functions

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Mean | `df["col"].mean()` | `pl.col("col").mean()` |
| Sum | `df["col"].sum()` | `pl.col("col").sum()` |
| Min | `df["col"].min()` | `pl.col("col").min()` |
| Max | `df["col"].max()` | `pl.col("col").max()` |
| Std | `df["col"].std()` | `pl.col("col").std()` |
| Count | `df["col"].count()` | `pl.col("col").count()` |
| Unique count | `df["col"].nunique()` | `pl.col("col").n_unique()` |
| First | `df["col"].iloc[0]` | `pl.col("col").first()` |
| Last | `df["col"].iloc[-1]` | `pl.col("col").last()` |

**Key difference**: In Pandas, these return a scalar value. In Polars, they return expressions that can be combined with other expressions in a single `select()` call.

### 6.4 String Expressions

Both Pandas and Polars provide a `.str` accessor (also called a "namespace") for string operations. This is one area where the two libraries are conceptually similar, but with important differences.

#### What is the `.str` Namespace?

The `.str` namespace is a collection of string methods that can be applied to text data. Instead of writing `upper(column)`, you write `column.str.to_uppercase()`. This keeps all string operations organized under one umbrella.

#### Similarities Between Pandas and Polars `.str`

| Aspect | Both Libraries |
|--------|----------------|
| Access pattern | Use `.str.method_name()` syntax |
| Chaining | Methods can be chained: `.str.lower().str.strip_chars()` |
| Vectorized | Operations apply to all values at once (no loops needed) |
| Null handling | Both handle null/NaN values gracefully |

#### Key Differences

| Aspect | Pandas `.str` | Polars `.str` |
|--------|---------------|---------------|
| **Applied to** | Series directly: `df["col"].str.upper()` | Expressions: `pl.col("col").str.to_uppercase()` |
| **Execution** | Immediate | Deferred (part of expression) |
| **Method names** | Shorter: `upper()`, `lower()`, `len()` | More explicit: `to_uppercase()`, `to_lowercase()`, `len_chars()` |
| **Return type** | Series (data) | Expression (recipe) |
| **After split** | Index with `.str[0]` | Use `.list.first()` or `.list.get(0)` |

#### Why Polars Uses Different Names

Polars chose more explicit method names to avoid ambiguity:

- `len()` vs `len_chars()`: In Polars, `len_chars()` counts characters, while `len_bytes()` counts bytes. This matters for Unicode text (e.g., "café" has 4 characters but 5 bytes in UTF-8).
- `upper()` vs `to_uppercase()`: The `to_` prefix makes it clear a transformation is happening.
- `startswith()` vs `starts_with()`: Polars uses snake_case consistently.

In [30]:
# String operations via .str namespace
employees.select(
    pl.col("first_name"),
    pl.col("first_name").str.to_uppercase().alias("name_upper"),
    pl.col("first_name").str.len_chars().alias("name_length")
)

first_name,name_upper,name_length
str,str,u32
"""Noah""","""NOAH""",4
"""Michael""","""MICHAEL""",7
"""Sophia""","""SOPHIA""",6
"""Luna""","""LUNA""",4
"""Gianna""","""GIANNA""",6
…,…,…
"""Ava""","""AVA""",3
"""Amelia""","""AMELIA""",6
"""Isabella""","""ISABELLA""",8


In [31]:
# Combine first and last name
employees.select(
    pl.col("first_name"),
    pl.col("last_name"),
    (pl.col("first_name") + " " + pl.col("last_name")).alias("full_name")
)

first_name,last_name,full_name
str,str,str
"""Noah""","""Smith""","""Noah Smith"""
"""Michael""","""Moore""","""Michael Moore"""
"""Sophia""","""Davis""","""Sophia Davis"""
"""Luna""","""Moore""","""Luna Moore"""
"""Gianna""","""Garcia""","""Gianna Garcia"""
…,…,…
"""Ava""","""Anderson""","""Ava Anderson"""
"""Amelia""","""Hernandez""","""Amelia Hernandez"""
"""Isabella""","""Williams""","""Isabella Williams"""


### Pandas Comparison: String Operations

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Uppercase | `df["col"].str.upper()` | `pl.col("col").str.to_uppercase()` |
| Lowercase | `df["col"].str.lower()` | `pl.col("col").str.to_lowercase()` |
| Length | `df["col"].str.len()` | `pl.col("col").str.len_chars()` |
| Contains | `df["col"].str.contains("x")` | `pl.col("col").str.contains("x")` |
| Replace | `df["col"].str.replace("a", "b")` | `pl.col("col").str.replace("a", "b")` |
| Split | `df["col"].str.split(",")` | `pl.col("col").str.split(",")` |
| Strip whitespace | `df["col"].str.strip()` | `pl.col("col").str.strip_chars()` |
| Starts with | `df["col"].str.startswith("x")` | `pl.col("col").str.starts_with("x")` |
| Ends with | `df["col"].str.endswith("x")` | `pl.col("col").str.ends_with("x")` |
| Concatenate | `df["a"] + " " + df["b"]` | `pl.col("a") + " " + pl.col("b")` |

**Note**: Polars uses `to_uppercase()`/`to_lowercase()` instead of `upper()`/`lower()`, and `len_chars()` instead of `len()` (to distinguish from byte length).

### 6.5 Boolean and Comparison Expressions

Boolean expressions are essential for filtering (covered in Session 2) and conditional logic.

In [32]:
# Comparison expressions return boolean values
employees.select(
    pl.col("first_name"),
    pl.col("salary"),
    (pl.col("salary") > 100000).alias("high_earner"),
    (pl.col("salary") >= 80000).alias("above_80k"),
    (pl.col("department") == "Engineering").alias("is_engineering"),
    (pl.col("department") != "Sales").alias("not_sales"),
)

first_name,salary,high_earner,above_80k,is_engineering,not_sales
str,i64,bool,bool,bool,bool
"""Noah""",77098,false,false,false,true
"""Michael""",122397,true,true,true,true
"""Sophia""",123907,true,true,true,true
"""Luna""",73893,false,false,false,true
"""Gianna""",89597,false,true,true,true
…,…,…,…,…,…
"""Ava""",51590,false,false,false,true
"""Amelia""",144775,true,true,false,true
"""Isabella""",47540,false,false,false,true


In [33]:
# Combining boolean expressions with AND (&) and OR (|)
employees.select(
    pl.col("first_name"),
    pl.col("department"),
    pl.col("salary"),
    # AND: both conditions must be true
    ((pl.col("department") == "Engineering") & (pl.col("salary") > 100000)).alias("eng_high_earner"),
    # OR: either condition can be true
    ((pl.col("department") == "Sales") | (pl.col("department") == "Marketing")).alias("sales_or_marketing"),
    # NOT: negate a condition
    (~(pl.col("is_active"))).alias("is_inactive"),
)

first_name,department,salary,eng_high_earner,sales_or_marketing,is_inactive
str,str,i64,bool,bool,bool
"""Noah""","""Operations""",77098,false,false,false
"""Michael""","""Engineering""",122397,true,false,false
"""Sophia""","""Engineering""",123907,true,false,false
"""Luna""","""Operations""",73893,false,false,false
"""Gianna""","""Engineering""",89597,false,false,false
…,…,…,…,…,…
"""Ava""","""HR""",51590,false,false,false
"""Amelia""","""Marketing""",144775,false,true,false
"""Isabella""","""HR""",47540,false,false,false


### Pandas Comparison: Boolean and Comparison Operations

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Equal | `df["col"] == value` | `pl.col("col") == value` |
| Not equal | `df["col"] != value` | `pl.col("col") != value` |
| Greater than | `df["col"] > value` | `pl.col("col") > value` |
| Less than | `df["col"] < value` | `pl.col("col") < value` |
| Greater or equal | `df["col"] >= value` | `pl.col("col") >= value` |
| Less or equal | `df["col"] <= value` | `pl.col("col") <= value` |
| AND | `(cond1) & (cond2)` | `(cond1) & (cond2)` |
| OR | `(cond1) \| (cond2)` | `(cond1) \| (cond2)` |
| NOT | `~condition` | `~condition` |
| Is null | `df["col"].isna()` | `pl.col("col").is_null()` |
| Is not null | `df["col"].notna()` | `pl.col("col").is_not_null()` |
| Is in list | `df["col"].isin([...])` | `pl.col("col").is_in([...])` |
| Between | `df["col"].between(a, b)` | `pl.col("col").is_between(a, b)` |

**Important**: Always wrap conditions in parentheses when using `&` and `|` due to Python's operator precedence.

## 7. Creating New Columns with `with_columns()`

To add new columns to an existing DataFrame, use `with_columns()`.

In [34]:
# Add new columns
employees_enhanced = employees.with_columns(
    # Annual bonus (10% of salary)
    (pl.col("salary") * 0.10).alias("bonus"),
    
    # Full name
    (pl.col("first_name") + " " + pl.col("last_name")).alias("full_name"),
    
    # Uppercase department
    pl.col("department").str.to_uppercase().alias("department_upper")
)

employees_enhanced.head()

employee_id,first_name,last_name,email,department,position,salary,hire_date,is_active,bonus,full_name,department_upper
i64,str,str,str,str,str,i64,str,bool,f64,str,str
1,"""Noah""","""Smith""","""employee1@company.com""","""Operations""","""Business Analyst""",77098,"""2020-07-03""",true,7709.8,"""Noah Smith""","""OPERATIONS"""
2,"""Michael""","""Moore""","""employee2@company.com""","""Engineering""","""Software Engineer""",122397,"""2022-09-25""",true,12239.7,"""Michael Moore""","""ENGINEERING"""
3,"""Sophia""","""Davis""","""employee3@company.com""","""Engineering""","""Tech Lead""",123907,"""2018-04-19""",true,12390.7,"""Sophia Davis""","""ENGINEERING"""
4,"""Luna""","""Moore""","""employee4@company.com""","""Operations""","""Process Engineer""",73893,"""2023-01-14""",true,7389.3,"""Luna Moore""","""OPERATIONS"""
5,"""Gianna""","""Garcia""","""employee5@company.com""","""Engineering""","""QA Engineer""",89597,"""2021-02-12""",true,8959.7,"""Gianna Garcia""","""ENGINEERING"""


### Pandas Comparison

```python
# Pandas way (modifies in place or requires copy)
df["bonus"] = df["salary"] * 0.10
df["full_name"] = df["first_name"] + " " + df["last_name"]

# Or with .assign() (returns new DataFrame)
df = df.assign(
    bonus=df["salary"] * 0.10,
    full_name=df["first_name"] + " " + df["last_name"]
)
```

Polars' `with_columns()` always returns a new DataFrame, promoting immutability.

## 8. Key Differences from Pandas

### No Index
Polars doesn't have a row index. This simplifies many operations and avoids index alignment issues.

### Expressions vs Direct Operations
Polars encourages using expressions (`pl.col()`) rather than direct column access.

### Immutability
Polars operations return new DataFrames rather than modifying in place.

### Strict Typing
Polars is stricter about data types, which helps catch errors early.

## Summary: Pandas to Polars Cheat Sheet

| Operation | Pandas | Polars |
|-----------|--------|--------|
| Import | `import pandas as pd` | `import polars as pl` |
| Create DataFrame | `pd.DataFrame({...})` | `pl.DataFrame({...})` |
| Read CSV | `pd.read_csv("file.csv")` | `pl.read_csv("file.csv")` |
| Write CSV | `df.to_csv("file.csv")` | `df.write_csv("file.csv")` |
| Select columns | `df[["col1", "col2"]]` | `df.select("col1", "col2")` |
| Add column | `df["new"] = expr` | `df.with_columns(expr.alias("new"))` |
| Column reference | `df["col"]` | `pl.col("col")` |
| Rename | `df.rename(columns={...})` | `df.rename({...})` |
| Shape | `df.shape` | `df.shape` |
| Data types | `df.dtypes` | `df.dtypes` or `df.schema` |

## Practice Exercises

Try these exercises using the `employees` DataFrame:

1. Select only the `position` and `salary` columns
2. Create a new column `monthly_salary` that divides `salary` by 12
3. Create a column `email_domain` that extracts the domain (everything after `@`) from the email addresses
4. Calculate the average, min, and max salary in a single `select()` statement
5. Select all columns that have the `String` data type
6. Create a boolean column `senior` that is `True` when `position` contains the word `"Senior"`
7. Using `with_columns()`, add **three** columns at once: `full_name` (first + last), `salary_k` (salary divided by 1000, rounded), and `name_length` (character count of `first_name`)
8. Use `describe()` to get summary statistics, then write a separate `select()` that computes the number of unique departments and the number of unique positions
9. Create a brand-new DataFrame from a dictionary with 5 rows and 3 columns of your choice, then write it to a Parquet file and read it back
10. Select only the `first_name` and `salary` columns, but rename them to `name` and `annual_pay` in the same `select()` call using `.alias()`

In [35]:
# Exercise 1: Select position and salary


In [36]:
# Exercise 2: Create monthly_salary column


In [37]:
# Exercise 3: Extract email domain


In [38]:
# Exercise 4: Calculate salary statistics


In [ ]:
# Exercise 5: Select all String columns


In [ ]:
# Exercise 6: Create a boolean "senior" column


In [ ]:
# Exercise 7: Add three columns at once with with_columns()


In [ ]:
# Exercise 8: describe() + count unique departments and positions


In [ ]:
# Exercise 9: Create a DataFrame, write to Parquet, read it back


In [ ]:
# Exercise 10: Select and rename columns with .alias()


## Next Session Preview

In Session 2, we'll dive deeper into:
- Filtering rows with `filter()`
- Conditional logic with `when().then().otherwise()`
- Groupby operations and aggregations
- Joining DataFrames
- Handling missing data